In [ ]:
import sys
import os

# Add project root to sys.path
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data_utils.preprocessing import *



config = PreprocessingConfig()

train = load_train("../data/train.csv")
store = load_store("../data/store.csv")

train = clean_sales_dataframe(train, config)
store = clean_store(store)

train = merge_sales_store(train, store, config)
train = add_calendar_features(train)

train = add_lag_features(train, "Store", ["Sales", "Customers"], config.lags)

# Apply hardcoded transforms decided from training EDA
train = apply_hardcoded_transforms(train)

# Select numeric columns and scale
numeric_cols = train.select_dtypes(include="float64").columns
train_scaled, x_scaler, y_scaler = scale_train(train, numeric_cols, target_col="sqrt_Sales")

FileNotFoundError: [Errno 2] No such file or directory: 'data/train.csv'

In [4]:
train.head()

,Store,DayOfWeek,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,...,Sales_lag1,Customers_lag1,Sales_lag7,Customers_lag7,Sales_lag14,Customers_lag14,Sales_lag28,Customers_lag28,Sales_lag365,Customers_lag365
Date,,,,,,,,,,,,,,,,,,,,,
2013-01-01,1115,2,0.0,0.0,0,0,1,1,d,c,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013-01-01,379,2,0.0,0.0,0,0,1,1,d,a,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013-01-01,378,2,0.0,0.0,0,0,1,1,a,c,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013-01-01,377,2,0.0,0.0,0,0,1,1,a,c,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013-01-01,376,2,0.0,0.0,0,0,1,1,a,a,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
train.describe()

,DayOfWeek,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,CompetitionDistance,CompetitionOpenSinceYear,Promo2,...,Sales_lag1,Customers_lag1,Sales_lag7,Customers_lag7,Sales_lag14,Customers_lag14,Sales_lag28,Customers_lag28,Sales_lag365,Customers_lag365
count,1.017209e+06,1.017209e+06,1.017209e+06,1.017209e+06,1.017209e+06,1.017209e+06,1.017209e+06,1.014567e+06,1.017209e+06,1.017209e+06,...,1.016094e+06,1.016094e+06,1.009404e+06,1.009404e+06,1.001599e+06,1.001599e+06,985989.000000,985989.000000,610234.000000,610234.000000
mean,3.998341e+00,5.773819e+03,6.331459e+02,8.301067e-01,3.815145e-01,3.052470e-02,1.786467e-01,5.430086e+03,1.369855e+03,5.005638e-01,...,5.770205e+03,6.328874e+02,5.765424e+03,6.328973e+02,5.770573e+03,6.333030e+02,5767.634690,633.294392,5680.841548,633.833208
std,1.997391e+00,3.849926e+03,4.644117e+02,3.755392e-01,4.857586e-01,1.720261e-01,3.830564e-01,7.715324e+03,9.358291e+02,4.999999e-01,...,3.849113e+03,4.644232e+02,3.847264e+03,4.645740e+02,3.852568e+03,4.649430e+02,3856.210118,465.310450,3823.296895,467.915971
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.000000e+01,-1.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000
25%,2.000000e+00,3.727000e+03,4.050000e+02,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,7.100000e+02,-1.000000e+00,0.000000e+00,...,3.724000e+03,4.040000e+02,3.722000e+03,4.040000e+02,3.722000e+03,4.040000e+02,3714.000000,404.000000,3636.000000,399.000000
50%,4.000000e+00,5.744000e+03,6.090000e+02,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.330000e+03,2.006000e+03,1.000000e+00,...,5.741000e+03,6.090000e+02,5.735000e+03,6.090000e+02,5.741000e+03,6.100000e+02,5738.000000,610.000000,5631.000000,609.000000
75%,6.000000e+00,7.856000e+03,8.370000e+02,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,6.890000e+03,2.011000e+03,1.000000e+00,...,7.851000e+03,8.370000e+02,7.844000e+03,8.370000e+02,7.854000e+03,8.380000e+02,7853.000000,838.000000,7731.000000,839.000000
max,7.000000e+00,4.155100e+04,7.388000e+03,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,7.586000e+04,2.015000e+03,1.000000e+00,...,4.155100e+04,7.388000e+03,4.155100e+04,7.388000e+03,4.155100e+04,7.388000e+03,41551.000000,7388.000000,38037.000000,7388.000000


In [ ]:
test = load_test("../data/test.csv")
test = clean_sales_dataframe(test, config)
test = merge_sales_store(test, store, config)
test = add_calendar_features(test)
test = add_lag_features(test, "Store", ["Sales", "Customers"], config.lags)
test = apply_hardcoded_transforms(test)

test.head()

,Id,Store,DayOfWeek,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,...,Promo2_Interval_in_Jul,Promo2_Interval_in_Jun,Promo2_Interval_in_Mar,Promo2_Interval_in_May,Promo2_Interval_in_Nov,Promo2_Interval_in_Oct,Promo2_Interval_in_Sept,Month,Year,Day
Date,,,,,,,,,,,,,,,,,,,,,
2015-08-01,41088,1115,6,1.0,0,0,1,d,c,5350.0,...,0,1,1,0,0,0,1,8,2015,1
2015-08-01,40523,378,6,1.0,0,0,0,a,c,2140.0,...,0,0,0,0,0,0,0,8,2015,1
2015-08-01,40522,377,6,1.0,0,0,0,a,c,100.0,...,0,0,0,1,1,0,0,8,2015,1
2015-08-01,40521,373,6,1.0,0,0,0,d,c,11120.0,...,1,0,0,0,0,1,0,8,2015,1
2015-08-01,40520,372,6,1.0,0,0,0,d,c,4880.0,...,1,0,0,0,0,1,0,8,2015,1


In [13]:
test.columns

Index(['Id', 'Store', 'DayOfWeek', 'Open', 'Promo', 'StateHoliday',
       'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance',
       'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek',
       'Promo2SinceYear', 'Competition_open_since_August',
       'Competition_open_since_December', 'Competition_open_since_February',
       'Competition_open_since_January', 'Competition_open_since_July',
       'Competition_open_since_June', 'Competition_open_since_March',
       'Competition_open_since_May', 'Competition_open_since_November',
       'Competition_open_since_October', 'Competition_open_since_September',
       'Promo2_Interval_in_Aug', 'Promo2_Interval_in_Dec',
       'Promo2_Interval_in_Feb', 'Promo2_Interval_in_Jan',
       'Promo2_Interval_in_Jul', 'Promo2_Interval_in_Jun',
       'Promo2_Interval_in_Mar', 'Promo2_Interval_in_May',
       'Promo2_Interval_in_Nov', 'Promo2_Interval_in_Oct',
       'Promo2_Interval_in_Sept', 'Month', 'Year', 'Day'],
      dtype='